# Breakout 1B · Targeting: from signals to a score

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

You look at two years of customs declarations the way a risk unit does: six **signals** (things
that look unusual about a line), a **ladder** of four ways to rank lines (a rule, distance from
normal, an oddness score, a learning score), two new ideas, and an exercise where your group
chooses five lines to inspect.

**How to use it.** Press ▶ on each cell from top to bottom, or *Runtime › Run all*. Change only
the sliders and drop-down menus. The code is folded: double-click a title to see it. Every
result ends with a **Finding** sentence.

Everything here is synthetic: every trader and name is generated.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v6/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v6"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import ladder, newideas, peer, signals
nb.use_style()
pd.set_option("display.max_colwidth", 40)
T = load_tables(DATA_URL)
print(f"Loaded {len(T)} tables, {len(T['declarations']):,} declaration lines. Ready.")

In [ ]:
#@title 0 · Prepare the tables
d, tariff, traders = T["declarations"], T["tariff"], T["traders"]
imp = d[(d["flow"] == "import") & (d["procedure"] == "IM4")].copy()
name = traders.set_index("trader_ref")["trader_id"]
name = name.where(name != "", traders.set_index("trader_ref").index.to_series())  # no TIN: customs' own code
imp["importer"] = imp["trader_ref"].map(name)
sheet = pd.DataFrame(load_json(DATA_URL, "worksheet_1b")["lines"]).set_index("line_ref")
print(f"{len(imp):,} import lines for home use; the worksheet holds {len(sheet)} lines.")

## 1 · Look at the data

In [ ]:
#@title 1 · A first look at the declarations
nb.steps("What does a declaration line hold?",
         "Take a sample of 1,000 lines.", "Show ten with the fields an officer reads first.")
sample = d.sample(1000, random_state=1)
nb.show(sample[["month", "trader_id", "ahtn8", "description", "origin_country", "customs_value_lcu", "net_weight_kg", "unit"]].head(10),
        formats={"customs_value_lcu": "{:,.0f}", "net_weight_kg": "{:,.1f}"})
nb.finding(f"{len(d):,} lines from {d['trader_ref'].nunique()} traders over {d['month'].nunique()} months; "
           f"{len(imp) / len(d):.0%} are imports for home use, the lines this session ranks.")

In [ ]:
#@title 1 · Five units, one weight: does the weight match the quantity?
nb.steps("Do the quantity and the net weight tell the same story?",
         "Work out the weight each line should have from its quantity and unit (tonnes, cartons, units, pallets).",
         "Divide the declared net weight by that expected weight.",
         "A line far above 1 has a weight keyed in the wrong unit.")
ratio = imp["net_weight_kg"] / signals.expected_weight(imp, tariff)
worst = imp.assign(weight_ratio=ratio).sort_values("weight_ratio", ascending=False).head(5)
nb.show(worst[["importer", "ahtn8", "quantity", "unit", "net_weight_kg", "weight_ratio"]],
        formats={"net_weight_kg": "{:,.0f}", "weight_ratio": "{:,.0f}x"})
top = worst.iloc[0]
nb.finding(f"One line declares {top.quantity:g} {top.unit} but a net weight of {top.net_weight_kg:,.0f} kg, "
           f"{top.weight_ratio:,.0f} times what the quantity implies: the weight was keyed in the wrong unit, "
           f"so its value per kilo looks {top.weight_ratio:,.0f} times too low. Check units before prices.")

In [ ]:
#@title 1 · Where the lines go today: channels
nb.steps("How do today's rules route import lines?", "Count lines by channel (green, yellow, red).")
counts = imp["channel"].value_counts().reindex(["green", "yellow", "red"])
fig, ax = nb.chart(f"{counts['red'] / counts.sum():.0%} of import lines go to the red channel")
ax.bar(counts.index, counts.values, color=[nb.COLORS["muted"], nb.COLORS["pale"], nb.COLORS["accent"]])
ax.set_ylabel("import lines")
plt.show()
share = imp.groupby("channel")["inspected"].mean()
nb.finding(f"Red lines are all inspected, yellow ones {share['yellow']:.0%} of the time, and a random "
           f"{imp['random_sample'].mean():.0%} of all lines is drawn before the rules run.")

## 2 · Signals

A **signal** is one thing that looks unusual about a line. Each cell computes one signal for every
line, shows the importers or lines where it is strongest, and says what it found.

In [ ]:
#@title 2 · The peer group: who do we compare a line with?
LINE = "W01"  #@param ["W01","W02","W03","W04","W05","W06","W07","W08","W09","W10","W11","W12","W13","W14","W15","W16","W17","W18","W19","W20"]
MIN_LINES = 30  #@param {type:"slider", min:10, max:60, step:5}
nb.steps("Which lines count as 'the same goods' for this worksheet line?",
         "Level 1: same 8-digit code and origin, last 12 months. Level 2: same code, any origin.",
         "Level 3: same 6-digit heading and origin. Level 4: same heading, any origin.",
         f"Use the first level with at least {MIN_LINES} lines.")
def find_line(ref):
    row = sheet.loc[ref]
    return imp[(imp["decl_id"] == row["decl_id"]) & (imp["line_no"] == row["line_no"])].iloc[0]
levels = peer.levels_for_line(find_line(LINE), imp, min_lines=MIN_LINES)
nb.show(levels, formats={"typical_value_per_kg": "{:,.1f}"}, highlight=levels["used"])
# The worksheet line whose verdict changes between level 1 and level 3
flips = []
for ref in sheet.index:
    line = imp.loc[[find_line(ref).name]]
    lvl1 = peer.peer_values(line, imp, min_lines=MIN_LINES, fixed_level=1)["ratio_to_peers"].iloc[0]
    lvl3 = peer.peer_values(line, imp, min_lines=MIN_LINES, fixed_level=3)["ratio_to_peers"].iloc[0]
    if np.isnan(lvl1) and lvl3 < 0.7:
        flips.append((ref, lvl3))
if flips:
    ref, lvl3 = flips[0]
    nb.finding(f"Line {ref} has no peers at level 1 (too few lines for its goods and origin), so a "
               f"level-1 check says nothing; at level 3 it sits at {lvl3:.0%} of typical, clearly low.")
else:
    nb.finding(f"With a minimum of {MIN_LINES} lines no worksheet line changes verdict between level 1 and level 3; "
               "move the slider back to 30.")

In [ ]:
#@title 2 · Price against peers: who is low again and again?
nb.steps("Which importers declare low values on many lines, not once?",
         "For every line, divide its value per kilo by the typical (median) value of its peer group.",
         "Call a line low when it is under 70% of typical.",
         "For each importer with 15 lines or more, take the share of its lines that are low.")
peers = peer.peer_values(imp, imp)
judged = peers["ratio_to_peers"].notna()
low = (peers.loc[judged, "ratio_to_peers"] < 0.70).groupby(imp.loc[judged, "importer"]).agg(["mean", "size"])
low = low[low["size"] >= 15].sort_values("mean", ascending=False)
table = low.head(8).rename(columns={"mean": "share_low", "size": "lines"})
table["median_ratio"] = peers.loc[judged].groupby(imp.loc[judged, "importer"])["ratio_to_peers"].median().reindex(table.index)
nb.show(table, formats={"share_low": "{:.0%}", "median_ratio": "{:.2f}"})
strong = table[table["share_low"] >= 0.5]
nb.finding(f"{len(strong)} importers declare at least half their lines below 70% of typical "
           f"({', '.join(strong.index[:4])}); for the median importer the share is "
           f"{low['mean'].median():.0%}. Repeated low prices point to a practice, not a one-off discount.")

In [ ]:
#@title 2 · Importer tenure: new names that grow fast
nb.steps("Which importers are both new and large?",
         "Count months since each importer registered with customs.",
         "Add its import value over the last 12 months and its typical price ratio.")
ten = signals.tenure(imp, traders)
ten["median_ratio"] = peers.groupby(imp["trader_ref"])["ratio_to_peers"].median().reindex(ten.index)
new = ten[ten["new_importer"]].sort_values("import_value_12m", ascending=False)
new.index = new.index.map(name)
nb.show(new.head(5)[["months_registered", "import_value_12m", "median_ratio"]],
        formats={"import_value_12m": "{:,.0f}", "months_registered": "{:.0f}", "median_ratio": "{:.2f}"})
first = new.iloc[0]
rank = int((ten["import_value_12m"] > first["import_value_12m"]).sum()) + 1
nb.finding(f"Importer {new.index[0]} registered {first['months_registered']:.0f} months ago and already ranks "
           f"#{rank} by import value, at {first['median_ratio']:.0%} of the typical price.")

In [ ]:
#@title 2 · Drift toward cheaper codes
nb.steps("Who is moving its goods toward a neighbouring code with a lower duty?",
         "List code pairs where the neighbouring code has a lower duty (tariff.duty_cliff_neighbour).",
         "For each importer and quarter, take the share of its pair lines declared under the cheaper code.",
         "Fit the trend across quarters (points per quarter); compare this quarter with the four before.")
drift = signals.code_drift(imp, tariff)
drift = drift[drift["lines"] >= 12]
view = drift.head(8)[["lines", "share_previous_four", "share_this_quarter", "trend_points"]]
view.index = view.index.map(name)
nb.show(view, formats={"share_previous_four": "{:.0%}", "share_this_quarter": "{:.0%}", "trend_points": "{:+.0f}"})
movers = drift[drift["trend_points"] >= 10]
nb.finding(f"{len(movers)} importers move toward the cheaper code by ten points a quarter or more "
           f"(fastest: {name[movers.index[0]]}, {movers['trend_points'].iloc[0]:+.0f} points a quarter). "
           "Line by line nothing looks wrong; over two years the shift is plain.")

In [ ]:
#@title 2 · Description against code
nb.steps("Does the description read like the code declared, or like a dearer one?",
         "Teach a reader which words go with which code, using every line.",
         "For each import line, score the chance its description belongs to a code with a higher duty.",
         "Read the top lines; descriptions too vague to read ('parts', 'goods') score 0.")
reader = signals.DescriptionModel.fit(d[d["flow"] != "transit"])
desc = reader.predict(imp, tariff)
top = imp.join(desc).sort_values("p_higher_duty_code", ascending=False).head(20)
nb.show(top.head(8)[["importer", "ahtn8", "description", "predicted_code", "p_higher_duty_code"]],
        formats={"p_higher_duty_code": "{:.0%}"})
neighbour = tariff.set_index("ahtn8")["duty_cliff_neighbour"]
pairs = int((top["predicted_code"].map(neighbour) == top["ahtn8"]).sum())
nb.finding(f"{pairs} of the 20 lines most likely to belong to a dearer code describe the goods of a code whose "
           "cheaper neighbour is the one declared: the value and weight look normal, only the words disagree.")

In [ ]:
#@title 2 · An origin that makes sense
nb.steps("Is a preference claimed from a country that actually makes the goods?",
         "Flag lines that claim a preference from an origin not in tariff.origin_producers.",
         "For each foreign seller, count the unrelated importers behind those doubtful claims.")
origin = signals.origin_check(imp, tariff, d[d["flow"] == "import"])
doubtful = imp[origin["doubtful_claim"]]
sellers = doubtful.groupby("exporter_id").agg(importers=("trader_ref", "nunique"), lines=("decl_id", "size"),
                                              origin=("origin_country", lambda s: s.mode().iloc[0]))
sellers = sellers.sort_values("importers", ascending=False)
nb.show(sellers.head(6))
two = sellers.head(2)
nb.finding(f"{len(doubtful)} lines claim a preference from an origin that does not make the goods; two sellers, "
           f"{two.index[0]} ({two['origin'].iloc[0]}) and {two.index[1]} ({two['origin'].iloc[1]}), are behind "
           f"{two['importers'].min()} or more unrelated importers each.")

In [ ]:
#@title 2 · Procedure and status history
nb.steps("Does an importer often change its declarations on its own?",
         "Leave out lines amended after an inspection finding.",
         "For each importer with 20 lines or more, take the share of lines amended or withdrawn.")
hist = signals.history(d)
hist = hist[hist["lines"] >= 20]
view = hist.head(5)[["lines", "amended_share", "withdrawn_share", "changed_share"]]
view.index = view.index.map(name)
nb.show(view, formats={"amended_share": "{:.0%}", "withdrawn_share": "{:.0%}", "changed_share": "{:.0%}"})
nb.finding(f"Importer {view.index[0]} amends or withdraws {view['changed_share'].iloc[0]:.0%} of its lines "
           f"on its own, against {hist['changed_share'].median():.0%} for the median importer.")

In [ ]:
#@title 2 · Twenty lines, six signals
nb.steps("How do the six signals look on the worksheet's twenty lines?",
         "Price, own history, description and origin per line; new importer and history per importer.")
keys = list(zip(sheet["decl_id"], sheet["line_no"]))
rows = imp.set_index(["decl_id", "line_no"]).loc[keys]
own = signals.own_history_z(rows.reset_index(), imp)
six = pd.DataFrame({
    "price": sheet["signal_price"].to_numpy(),
    "own_history": (-own).clip(lower=0).round(1).to_numpy(),
    "description": sheet["signal_description"].to_numpy(),
    "origin": sheet["signal_origin"].to_numpy(),
    "new_importer": sheet["new_importer"].astype(int).to_numpy(),
    "history": rows["trader_ref"].map(signals.history(d)["changed_share"]).round(2).to_numpy(),
}, index=sheet.index)
strong = (six[["price", "description", "origin"]] >= 0.4).any(axis=1) | (six["own_history"] >= 3)
nb.show(six, formats={c: "{:.2f}" for c in ["price", "description", "origin", "history"]}, highlight=strong)
nb.finding(f"{strong.sum()} of the 20 lines show at least one strong signal; no single signal finds them all, "
           "which is why the ladder combines them.")

## 3 · The ladder

Four ways to rank lines, each a step up from the one before: a **rule**, **distance from normal**,
an **oddness score**, and a **learning score**. Each step keeps what the previous one does well
and fixes one of its mistakes.

In [ ]:
#@title 3 · Step 1, a rule: the cheapest tenth
PERCENTILE = 10  #@param {type:"slider", min:5, max:30, step:5}
nb.steps("Which worksheet lines would a simple price rule send to inspection?",
         f"Find the value below which the cheapest {PERCENTILE}% of all import lines sit, relative to their peers.",
         "Flag the worksheet lines under that cut-off.")
cut = np.nanpercentile(peers["ratio_to_peers"], PERCENTILE)
ratio_sheet = 1 - sheet["signal_price"]
flagged = ratio_sheet <= cut
view = pd.DataFrame({"ratio_to_peers": ratio_sheet, "flagged": flagged}).sort_values("ratio_to_peers")
nb.show(view.head(10), formats={"ratio_to_peers": "{:.2f}"}, highlight=view["flagged"])
nb.finding(f"The rule flags {flagged.sum()} of the 20 lines ({', '.join(view.index[view['flagged']])}). "
           "It sees prices, not reasons: a unit error and a contracted discount look as low as a finding.")

In [ ]:
#@title 3 · Step 2, distance from normal: against peers and against itself
Z = -2.5  #@param {type:"slider", min:-4, max:-1.5, step:0.5}
nb.steps("Is a low line far from normal, for its peers and for the importer itself?",
         "Distance from the peers: how many typical spreads (median and MAD) below the peer median.",
         "Distance from itself: the same, against the importer's own earlier lines in the same code.",
         f"Flag lines below {Z} spreads.")
recent = imp[imp["month"] >= "2026-04"]
normal = recent[peers.loc[recent.index, "ratio_to_peers"].between(0.8, 1.25)]
own_z = signals.own_history_z(normal, imp[imp["month"] < "2026-04"]).dropna()
lowest = normal.loc[own_z.sort_values().index[:6]].assign(own_z=own_z)
nb.show(lowest[["importer", "month", "ahtn8", "own_z"]].assign(ratio_to_peers=peers.loc[lowest.index, "ratio_to_peers"]),
        formats={"own_z": "{:.1f}", "ratio_to_peers": "{:.2f}"})
who = lowest["importer"].iloc[0]
nb.finding(f"Importer {who} looks normal against its peers but sits {abs(lowest['own_z'].iloc[0]):.0f} spreads below "
           "its own record since April 2026: a premium buyer now declaring at the average.")

In [ ]:
#@title 3 · Step 3, an oddness score with its reasons
nb.steps("Which worksheet lines are odd on several counts at once?",
         "Score price, description and origin together against all import lines (an isolation forest): "
         "a line is odd when few lines look like it.",
         "For each line, the signals that pushed its score up are its reasons.")
signal_cols = ["signal_price", "signal_description", "signal_origin"]
population = pd.DataFrame({"signal_price": (1 - peers["ratio_to_peers"]).clip(0, 1),
                           "signal_description": desc["p_higher_duty_code"],
                           "signal_origin": origin["doubtful_claim"].astype(float)})
odd = ladder.step3_oddness(sheet[signal_cols], fit_on=population).sort_values("oddness", ascending=False)
odd["reasons"] = odd["reasons"].map(lambda r: ", ".join(r))
nb.show(odd.head(8), formats={"oddness": "{:.2f}"})
two = ((sheet["signal_price"] >= 0.3).astype(int) + (sheet["signal_description"] >= 0.3) + (sheet["signal_origin"] >= 1))
mixed = [ref for ref in odd.index[:3] if two[ref] >= 2]
if mixed:
    ref = mixed[0]
    nb.finding(f"Line {ref} is among the three oddest ({odd.at[ref, 'reasons']}): {1 - sheet.at[ref, 'signal_price']:.0%} of "
               f"the typical price and a description that reads like a dearer code ({sheet.at[ref, 'signal_description']:.0%}), "
               "two signals that each check alone ranks in the middle.")
else:
    nb.finding(f"The oddest line is {odd.index[0]} ({odd['reasons'].iloc[0]}).")

In [ ]:
#@title 3 · Step 4, a learning score, and why the random draw matters
BUDGET = 5  #@param {type:"slider", min:1, max:10, step:1}
nb.steps("Does a model trained on past inspections find more than the rules, at the same budget?",
         "Train on lines inspected before January 2026, with their outcomes.",
         f"Score the lines from January 2026; among those opened at random, take the top {BUDGET}% by each method.",
         "Retrain without the random draw and compare.")
features = pd.DataFrame({
    "price_gap": (1 - peers["ratio_to_peers"]).clip(0, 1), "p_higher_duty_code": desc["p_higher_duty_code"],
    "doubtful_claim": origin["doubtful_claim"].astype(float), "new_importer": imp["new_importer"].astype(float)})
train = (imp["month"] < "2026-01") & imp["inspected"]
test = imp["month"] >= "2026-01"
outcome = imp["inspection_outcome"] == "finding"
model, _ = ladder.learning_score(features[train], outcome[train], features[test])
biased, _ = ladder.learning_score(features[train & ~imp["random_sample"]], outcome[train & ~imp["random_sample"]], features[test])
rules = imp.loc[test, "channel"].map({"red": 2, "yellow": 1, "green": 0}) + features.loc[test, "price_gap"] * 0.01
# Only the lines opened at random have an outcome every method can be judged on fairly.
fair = imp.loc[test, "random_sample"]
rates = {name_: ladder.hit_rate_at_budget(score[fair], outcome[test][fair], BUDGET / 100)
         for name_, score in [("legacy rules", rules), ("learning score", model), ("trained without random draw", biased)]}
rates["random draw"] = outcome[test][fair].mean()
fig, ax = nb.chart(f"At a {BUDGET}% budget the learning score finds something on {rates['learning score']:.0%} of lines")
colors = [nb.COLORS["muted"], nb.COLORS["accent"], nb.COLORS["amber"], nb.COLORS["pale"]]
ax.bar(list(rates), [v * 100 for v in rates.values()], color=colors)
ax.set_ylabel("lines with a finding, %")
plt.show()
washed = origin["doubtful_claim"][test]
nb.finding(f"The learning score finds {rates['learning score']:.0%} against {rates['legacy rules']:.0%} for the legacy rules "
           f"and {rates['random draw']:.0%} for the random draw (in real administrations a random draw finds something on "
           f"1 to 8 lines in 100). Trained without the random draw it scores lines with a "
           f"doubtful origin at {biased[washed].mean():.0%} on average, against {model[washed].mean():.0%} with it: "
           "it never learns what the rules never opened.")

## 3b · Two new ideas

In [ ]:
#@title 3b · Groups inside one tariff line: 200 smartphone lines
K = 2  #@param {type:"slider", min:2, max:4, step:1}
nb.steps("Does one tariff line hide different products?",
         "Take the 200 smartphone lines (8517.13.00).", "Plot value per kilo against weight per unit.",
         f"Let the machine find {K} groups (k-means), then find the line farthest from every group.")
phones = pd.DataFrame(load_json(DATA_URL, "cluster_1b")["lines"])
groups = newideas.tariff_line_groups(phones, k=K)
far = groups["distance_to_nearest_group"].idxmax()
fig, ax = nb.chart("One line is priced like an entry model but packed like a premium one")
palette = [nb.COLORS["accent"], nb.COLORS["muted"], nb.COLORS["pale"], nb.COLORS["navy2"]]
for g, part in groups.groupby("group"):
    ax.scatter(part["kg_per_unit"], part["usd_per_kg"], s=14, color=palette[g - 1], label=f"group {g}")
ax.scatter(groups.at[far, "kg_per_unit"], groups.at[far, "usd_per_kg"], s=220, facecolors="none",
           edgecolors=nb.COLORS["amber"], linewidths=2.5, label="fits no group")
ax.set_xlabel("weight per unit, kg"); ax.set_ylabel("value per kilo, USD"); ax.legend()
plt.show()
row = groups.loc[far]
nb.finding(f"Line {phones.at[far, 'decl_id']}-{phones.at[far, 'line_no']} is declared at {row['usd_per_kg']:.0f} USD/kg, "
           f"an entry-level price, with {row['kg_per_unit']:.3f} kg per unit, a premium model's weight: it fits "
           f"no group. With k={K} the groups are a choice, not a truth; the odd line stays odd.")

In [ ]:
#@title 3b · A trader against its own past
IMPORTER = "T376252"  #@param ["T376252", "T251410", "T773224", "T1145", "T552961"]

SHOW_REFERENCE = False  #@param {type:"boolean"}
nb.steps("Did an importer's price fall away from its own history?",
         "Build the importer's band from its earlier months only (median and spread, shifted one month).",
         "Flag the months below the band; a floor on the spread keeps the first months quiet.")
drift_sheet = load_json(DATA_URL, "drift_1b")["series"]
series = next(s for s in drift_sheet if s["importer"] == IMPORTER)
months = pd.DataFrame(series["months"])
fig, ax = nb.chart(f"Importer {series['importer']}: value per kilo against its own band")
ax.plot(months["month"], months["uv_lcu_per_kg"], color=nb.COLORS["accent"], label="value per kilo")
ax.plot(months["month"], months["band_lower"], color=nb.COLORS["muted"], linestyle=":", label="lower edge of its band")
below = months[months["below_band"]]
ax.scatter(below["month"], below["uv_lcu_per_kg"], color=nb.COLORS["amber"], zorder=3, label="below the band")
if SHOW_REFERENCE:
    ax.plot(months["month"], months["ref_median"], color=nb.COLORS["navy2"], linestyle="--", label="typical value, all importers")
ax.set_xticks(months["month"][::6]); ax.legend(); plt.show()
if len(below):
    first = below.iloc[0]
    depth = 1 - first["uv_lcu_per_kg"] / first["band_median"]
    nb.finding(f"Importer {series['importer']} falls below its own band in {first['month']}, {depth:.0%} under its usual "
               f"level, and stays there for {len(below)} months. Tick 'show reference' to see whether everyone fell too.")
else:
    nb.finding(f"Importer {series['importer']} stays inside its own band for all {len(months)} months.")

## 4 · Exercise: choose five lines to inspect

Roles: the **analyst** works the sliders, the **officer** decides, the **committee** writes what it
would sign. Weigh the three signals, read the ranked table, then pick five lines and lock them.
Nothing about outcomes is visible until the facilitator's reveal.

In [ ]:
#@title 4 · Weigh the signals and rank the twenty lines
HINTS = ["Not every low price is a finding.", "Weigh price, description and origin, then check each pick for a units or own-history explanation.", "Price alone catches two decoys; add description and origin weight, then drop the unit error and the discount."]

PRICE = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
DESCRIPTION = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
ORIGIN = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
score = PRICE * sheet["signal_price"] + DESCRIPTION * sheet["signal_description"] + ORIGIN * sheet["signal_origin"]
ranked = sheet.assign(score=score).sort_values("score", ascending=False)
nb.show(ranked[["importer", "ahtn8", "description", "signal_price", "signal_description", "signal_origin", "score"]],
        formats={"signal_price": "{:.2f}", "signal_description": "{:.2f}", "signal_origin": "{:.0f}", "score": "{:.2f}"})
nb.hint(HINT_LEVEL, HINTS, "1b.exercise")
nb.finding(f"With these weights the top five are {', '.join(ranked.index[:5])}. Before locking, check each pick for a "
           "units problem or a price that is normal for that importer.")

In [ ]:
#@title 4 · Pick five lines and lock them
#@markdown If the selection list does not show, type your five line references here instead.
MY_PICKS = ""  #@param {type:"string"}
import ipywidgets as widgets
from IPython.display import display
LOCKED_PICKS = [p.strip().upper() for p in MY_PICKS.split(",") if p.strip()]
choose = widgets.SelectMultiple(options=list(ranked.index), rows=10, description="Your five")
lock = widgets.Button(description="Lock my picks", button_style="primary")
note = widgets.Output()
def on_lock(_):
    global LOCKED_PICKS
    LOCKED_PICKS = list(choose.value)
    with note:
        note.clear_output()
        print(f"Locked {len(LOCKED_PICKS)} lines: {', '.join(LOCKED_PICKS)}" + ("" if len(LOCKED_PICKS) == 5 else "  (choose exactly five)"))
lock.on_click(on_lock)
display(widgets.VBox([choose, lock, note]))
if LOCKED_PICKS:
    print(f"Picks typed in: {', '.join(LOCKED_PICKS)}")

## 5 · Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 5 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "6mds3VnS4mGL7D4YKahxGQ==", "data": "cUTlnC6shRPgh1bty1CVFsVkigCDy3KiJhN2u8nR4DunuxTDMCXL931gxUVFcm4ruNRUdOKCtrIrHaPWcLLglDyn7MrJN2NWS2Z/xawy43A0qPnIcRMe3OoI4yqFHT8X+I12aZpfHu+hFL40cgDYMIi9zca/3nW0tLk809b9iUnGpLXAcQB6UQu70ptLAo1RBYZrtEg4tD5GoR/wX9vo9EWZl3x7pNi8zdCo9KsOFzJbn/AT8M1fYw/maZDtc2/QdDz64Dz8zJ9kX8FNJjCer0CgKejyRskqjo/fzaP3jqp9zTgrruiW4wHtuTmPLIzEDYFVTozGNeNcD6mrhXDgQ4IlCMfk6TTLP5wFVn0XUVYLJJmEns0nCFqlDdt83YB5Rq5T9ci8Bf+ZqAKtUXT0M3BRdzsTa91auOWKKP4XGA4DmEeV0BBAeFudXWSUMx9ZK3PWMJQg/pmTGpHTCHh3GaRuwTHzRV5/jnUsufkPPqU8JqfexJFv9+7CdK5g+4dKx8V6en2xEZP8CJlyNPiNzJ9Li99/C9MbYmM5cTwvuC9rAWtuisMIBh4ZmkBnehXmh6xLjUjD/GiUcWFhwLRfDAB43C99Arid5ZZhwSy2r/zV9/8Nuanaob3efKgboT62VLWGfIBVfP5XUIcIwpYYcbOJHw/iFDPYmWhMH3Wht7zr21y4eHVcUN421pikwWzjzcsN/R/gED2eKNexZzPcKN8ECo19pYH4aN/JTuaEkE5Zs55vzOcZq0+NG4QTMMUH2+eeWw==", "mac": "b257aa5a263ef18115f2c90e8e3fac60ab41fc1d192b50b287a41735f24738f9"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["lines"]).set_index("line_ref")
    model = pd.DataFrame(load_json(DATA_URL, "model_1b")["lines"]).set_index("line_ref")
    key["picked"] = key.index.isin(LOCKED_PICKS)
    nb.show(key[["outcome", "finding_type", "duty_recovered_lcu", "picked"]], formats={"duty_recovered_lcu": "{:,.0f}"},
            highlight=key["outcome"] == "finding")
    hits = key.loc[key["picked"], "outcome"].eq("finding").sum()
    duty = key.loc[key["picked"] & key["outcome"].eq("finding"), "duty_recovered_lcu"].sum()
    top5 = model.sort_values("score", ascending=False).head(5)
    reasons = {r: ", ".join(x["signal"] for x in v) for r, v in top5["reasons"].items()}
    nb.finding(f"Your group found {hits} of 5 (duty recovered LCU {duty:,.0f}); a random pick finds "
               f"{answer['random_benchmark_hits_of_5']} of 5 (random, expected: 30% on this enriched set; real-world: 1 to 8%). "
               "The model's top five: "
               + "; ".join(f"{r} ({reasons[r]})" for r in top5.index) + ".")
    for item in answer["explanations"]:
        print(f"- {item}")
    if nb.HINTS_USED:
        print("Hints used:", nb.HINTS_USED)

## 6 · For your analysts

The same pipeline with the code shown and the technical names. The four layers of the deck:

| Layer | In this notebook | Technical name |
| --- | --- | --- |
| Data | section 1, `load_tables` | extract, data dictionary |
| Signals | section 2 | features: robust ratio to a reference class, tenure, share drift, text classifier, origin plausibility, amendment rate |
| Score | section 3 | percentile rule, robust z-score (median and MAD), anomaly score (isolation forest), supervised model (gradient boosting) with feature contributions |
| Decision | section 4 | the inspection list, the budget, the outcome record (label) |

**Your own extract.** Point `DATA_URL` (section 0) at a folder holding the same CSV files, or rename
your columns first with a mapping such as
`{"DECL_NO": "decl_id", "ITEM_NO": "line_no", "HS_CODE": "ahtn8", "CIF_VALUE": "customs_value_lcu", "NET_MASS": "net_weight_kg"}`.
The methods are in the `workshop_methods` package: `peer`, `signals`, `ladder`, `newideas`.

**Keep a random share.** A model learns only from the lines someone chose to open. Draw the random
share before the rules run, record every outcome, and the committee decides its size.

**Further reading (titles only).** WCO Revenue Package; WCO Customs Risk Management Compendium;
IMF Revenue Administration Fiscal Information Tool (RA-FIT).

In [ ]:
#@title 6 · The whole pipeline in one cell
# 1. Data: home-use import lines
imports = d[(d["flow"] == "import") & (d["procedure"] == "IM4")]
# 2. Signals
peer_ratio = peer.peer_values(imports, imports)["ratio_to_peers"]           # reference class, median not mean
text = signals.DescriptionModel.fit(d[d["flow"] != "transit"]).predict(imports, tariff)
plausible = signals.origin_check(imports, tariff, d[d["flow"] == "import"])
X = pd.DataFrame({"price_gap": (1 - peer_ratio).clip(0, 1), "p_higher_duty_code": text["p_higher_duty_code"],
                  "doubtful_claim": plausible["doubtful_claim"].astype(float), "new_importer": imports["new_importer"].astype(float)})
# 3. Score: gradient boosting on past inspection outcomes, with three reasons per line
seen = imports["inspected"]
score, why = ladder.learning_score(X[seen], imports.loc[seen, "inspection_outcome"] == "finding", X)
# 4. Decision: the top 5% with their reasons
shortlist = imports.assign(score=score, reasons=why["reasons"]).nlargest(int(len(imports) * 0.05), "score")
print(shortlist[["decl_id", "line_no", "trader_id", "ahtn8", "score"]].head(10).to_string(index=False))